# A股量化预测系统分析笔记本

本笔记本用于数据分析、特征探索和模型评估。

In [ ]:
# 导入必要的库
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置显示
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

In [ ]:
# 加载配置
with open('../config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("配置加载完成")

## 1. 数据加载与探索

In [ ]:
from data.qlib_converter import SimpleDataLoader

# 加载数据
loader = SimpleDataLoader(config)
df = loader.load_data()

print(f"数据形状: {df.shape}")
print(f"时间范围: {df['date'].min()} 到 {df['date'].max()}")
print(f"股票数量: {df['code'].nunique()}")

In [ ]:
# 数据概览
df.head()

In [ ]:
# 数据统计
df.describe()

In [ ]:
# 缺失值统计
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

## 2. 特征工程

In [ ]:
from features import FeatureEngine

# 加载指数数据
from pathlib import Path
processed_dir = Path(config.get('paths', {}).get('processed_data_dir', './data_storage/processed'))
index_path = processed_dir / 'index_processed.parquet'
index_df = pd.read_parquet(index_path) if index_path.exists() else None

# 计算特征
feature_engine = FeatureEngine(config)
df_features = feature_engine.compute_all_features(df.copy(), index_df)

print(f"特征计算后形状: {df_features.shape}")

In [ ]:
# 特征相关性分析
feature_cols = [c for c in df_features.columns if c.startswith(('ma_', 'rsi_', 'macd_', 'alpha_'))]
sample_features = feature_cols[:20]  # 取前20个特征

if len(sample_features) > 0:
    corr = df_features[sample_features].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0)
    plt.title('特征相关性矩阵')
    plt.tight_layout()
    plt.show()

## 3. 模型训练与评估

In [ ]:
from data import DataProcessor
from models import LightGBMModel

# 数据划分
processor = DataProcessor(config)
train_df, valid_df, test_df = processor.get_train_valid_test_split(df_features)

print(f"训练集: {len(train_df)}")
print(f"验证集: {len(valid_df)}")
print(f"测试集: {len(test_df)}")

In [ ]:
# 获取特征列
exclude_cols = [
    'date', 'code', 'name', 'industry',
    'open', 'high', 'low', 'close', 'volume', 'amount',
    'turnover', 'pct_change', 'change', 'amplitude',
    'year', 'month', 'day', 'weekday', 'quarter',
    'is_month_start', 'is_month_end',
    'is_suspended', 'is_limit_up', 'is_limit_down', 'is_abnormal',
    'return_1d', 'return_5d', 'return_1d_rank', 'return_5d_rank',
    'label_binary_1d', 'label_binary_5d'
]

feature_cols = [c for c in df_features.columns if c not in exclude_cols]
feature_cols = [c for c in feature_cols if df_features[c].notna().sum() > len(df_features) * 0.5]

print(f"特征数量: {len(feature_cols)}")

In [ ]:
# 准备数据
label_col = 'return_1d_rank'

train_df = train_df.dropna(subset=feature_cols + [label_col])
valid_df = valid_df.dropna(subset=feature_cols + [label_col])
test_df = test_df.dropna(subset=feature_cols + [label_col])

train_df[feature_cols] = train_df[feature_cols].fillna(0)
valid_df[feature_cols] = valid_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

In [ ]:
# 训练LightGBM
lgb_model = LightGBMModel(config)
result = lgb_model.train(
    train_df[feature_cols], train_df[label_col], train_df['date'],
    valid_df[feature_cols], valid_df[label_col], valid_df['date']
)

print(f"最佳迭代: {result['best_iteration']}")

In [ ]:
# 特征重要性
importance = result['feature_importance']

plt.figure(figsize=(10, 12))
plt.barh(importance['feature'][:30], importance['importance'][:30])
plt.xlabel('重要性')
plt.title('Top 30 特征重要性')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. IC分析

In [ ]:
from backtest import BacktestEvaluator

# 预测
predictions = lgb_model.predict(test_df[feature_cols])

# 构建预测DataFrame
pred_df = test_df[['date', 'code']].copy()
pred_df['prediction'] = predictions

# 评估
evaluator = BacktestEvaluator(config)
report = evaluator.generate_report(
    pred_df,
    test_df[['date', 'code', 'return_1d']]
)

In [ ]:
# IC时序图
ic_df = report['ic_daily']

plt.figure(figsize=(14, 5))
plt.bar(pd.to_datetime(ic_df['date']), ic_df['ic'], alpha=0.7, width=1)
plt.axhline(y=ic_df['ic'].mean(), color='red', linestyle='--', label=f"Mean: {ic_df['ic'].mean():.4f}")
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('日期')
plt.ylabel('IC')
plt.title('每日IC')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# IC汇总
print("IC汇总统计:")
for k, v in report['ic_summary'].items():
    print(f"  {k}: {v:.4f}")

## 5. 分组收益分析

In [ ]:
# 分组累计收益
group_returns = report['group_returns']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 累计收益曲线
colors = plt.cm.RdYlGn(np.linspace(0, 1, 10))
ax1 = axes[0]
for i in range(1, 11):
    col = f'group_{i}'
    if col in group_returns.columns:
        cumret = (1 + group_returns[col].fillna(0)).cumprod()
        ax1.plot(cumret, label=f'Group {i}', color=colors[i-1])

ax1.set_title('分组累计收益')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# 多空组合
ax2 = axes[1]
if 'long_short' in group_returns.columns:
    ls_cumret = (1 + group_returns['long_short'].fillna(0)).cumprod()
    ax2.plot(ls_cumret, color='steelblue', linewidth=2)
    ax2.fill_between(range(len(ls_cumret)), 1, ls_cumret, alpha=0.3)
ax2.axhline(y=1, color='black', linestyle='--')
ax2.set_title('多空组合累计收益')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 分组汇总
print("分组收益汇总:")
print(report['group_summary'].to_string(index=False))

## 6. 生成今日预测

In [ ]:
# 获取最新数据的预测
latest_date = df_features['date'].max()
latest_df = df_features[df_features['date'] == latest_date].copy()
latest_df[feature_cols] = latest_df[feature_cols].fillna(0)

# 预测
latest_pred = lgb_model.predict(latest_df[feature_cols])
latest_df['prediction'] = latest_pred
latest_df['pred_rank'] = latest_df['prediction'].rank(pct=True)

# Top 20
top_20 = latest_df.nlargest(20, 'prediction')[['code', 'name', 'close', 'prediction', 'pred_rank']]
print(f"\n{latest_date} Top 20 股票预测:")
print(top_20.to_string(index=False))